In [0]:
# Day 2 - Customer Schema Validation

schema_contract = {
    "customer_id": "string",
    "customer_name": "string",
    "email": "string",
    "state": "string",
    "updated_at": "utc_timestamp"
}

required_columns = list(schema_contract.keys())

print("Expected customer schema:")
print(schema_contract)

In [0]:
import json

with open("../config/customer_schema.json", "r") as file:
    customer_schema = json.load(file)

print(customer_schema)

In [0]:
customer_source_path = "/Volumes/de_prac/day01/data-files/incoming/customers"

customer_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv(customer_source_path)
)

print("Source columns:")
print(customer_df.columns)

customer_df.printSchema()

display(customer_df.limit(5))

In [0]:
expected_columns = list(customer_schema["columns"].keys())
actual_columns = customer_df.columns

missing_columns = [
    col for col in expected_columns
    if col not in actual_columns
]

extra_columns = [
    col for col in actual_columns
    if col not in expected_columns
]

print("Expected columns:", expected_columns)
print("Actual columns:", actual_columns)
print("Missing columns:", missing_columns)
print("Extra columns:", extra_columns)

In [0]:
strict_mode = customer_schema["strict_mode"]

if missing_columns:
    validation_status = "FAIL"
    can_proceed_downstream = False
    failure_reason = f"Missing required columns: {missing_columns}"

elif strict_mode and extra_columns:
    validation_status = "FAIL"
    can_proceed_downstream = False
    failure_reason = f"Unexpected columns found in strict mode: {extra_columns}"

else:
    validation_status = "PASS"
    can_proceed_downstream = True
    failure_reason = None

print("Validation status:", validation_status)
print("Can proceed downstream:", can_proceed_downstream)
print("Failure reason:", failure_reason)

In [0]:
from pyspark.sql.functions import col, to_timestamp, trim

timestamp_check_df = customer_df.withColumn(
    "_parsed_updated_at",
    to_timestamp(col("updated_at"))
)

invalid_timestamp_count = (
    timestamp_check_df
    .filter(
        (trim(col("updated_at")) != "") &
        col("updated_at").isNotNull() &
        col("_parsed_updated_at").isNull()
    )
    .count()
)

print("Invalid timestamp count:", invalid_timestamp_count)

In [0]:
from collections import Counter
from pyspark.sql.functions import col, trim, expr


def validate_schema(df, schema_config):

    expected_columns = list(schema_config["columns"].keys())
    actual_columns = df.columns
    strict_mode = schema_config.get("strict_mode", False)

    # 1. Missing columns
    missing_columns = [
        c for c in expected_columns
        if c not in actual_columns
    ]

    # 2. Extra columns
    extra_columns = [
        c for c in actual_columns
        if c not in expected_columns
    ]

    # 3. Duplicate headers
    column_counts = Counter(actual_columns)

    duplicate_headers = [
        c for c, count in column_counts.items()
        if count > 1
    ]

    # 4. Datatype validation
    actual_types = dict(df.dtypes)
    datatype_mismatches = []

    for column_name, rules in schema_config["columns"].items():

        if column_name not in actual_columns:
            continue

        expected_type = rules["type"]

        if expected_type == "string":

            if actual_types[column_name] != "string":
                datatype_mismatches.append({
                    "column": column_name,
                    "expected": "string",
                    "actual": actual_types[column_name]
                })

        elif expected_type == "utc_timestamp":

            # Source value must remain a string.
            # Timestamp interpretation is checked separately.
            if actual_types[column_name] != "string":
                datatype_mismatches.append({
                    "column": column_name,
                    "expected": "string containing UTC timestamp",
                    "actual": actual_types[column_name]
                })

    # 5. Timestamp parse validation
    parse_failures = []

    if "updated_at" in actual_columns:

        invalid_timestamp_count = (
            df
            .withColumn(
                "_parsed_updated_at",
                expr("try_cast(updated_at AS TIMESTAMP)")
            )
            .filter(
                col("updated_at").isNotNull()
                & (trim(col("updated_at")) != "")
                & col("_parsed_updated_at").isNull()
            )
            .count()
        )

        if invalid_timestamp_count > 0:
            parse_failures.append({
                "column": "updated_at",
                "failure_count": invalid_timestamp_count
            })

    # 6. Build exact critical failure reasons
    critical_errors = []

    if missing_columns:
        critical_errors.append(
            f"Missing required columns: {missing_columns}"
        )

    if duplicate_headers:
        critical_errors.append(
            f"Duplicate headers found: {duplicate_headers}"
        )

    if datatype_mismatches:
        critical_errors.append(
            f"Datatype mismatches: {datatype_mismatches}"
        )

    if parse_failures:
        critical_errors.append(
            f"Timestamp parse failures: {parse_failures}"
        )

    if strict_mode and extra_columns:
        critical_errors.append(
            f"Unexpected columns found in strict mode: {extra_columns}"
        )

    # 7. Final critical failure decision
    critical_failure = bool(critical_errors)

    # 8. Structured validation report
    report = {
        "validation_status": "FAIL" if critical_failure else "PASS",
        "missing_columns": missing_columns,
        "extra_columns": extra_columns,
        "duplicate_headers": duplicate_headers,
        "datatype_mismatches": datatype_mismatches,
        "parse_failures": parse_failures,
        "critical_errors": critical_errors,
        "can_proceed_downstream": not critical_failure
    }

    return report

In [0]:
import csv

valid_test_path = "/Workspace/Users/lkongara31@gmail.com/data-engineering-practice/tests/day02/customers_valid.csv"

with open(valid_test_path, "r") as file:
    reader = csv.reader(file)
    rows = list(reader)

header = rows[0]
data = rows[1:]

valid_test_df = spark.createDataFrame(data, schema=header)

valid_test_report = validate_schema(
    valid_test_df,
    customer_schema
)

print(valid_test_report)

In [0]:
import os

print(os.getcwd())

In [0]:
import os

repo_root = "/Workspace/Users/lkongara31@gmail.com/data-engineering-practice"

print(os.listdir(repo_root))

In [0]:
tests_root = "/Workspace/Users/lkongara31@gmail.com/data-engineering-practice/tests"

print(os.listdir(tests_root))

In [0]:
missing_test_path = "/Workspace/Users/lkongara31@gmail.com/data-engineering-practice/tests/day02/customers_missing_column.csv"

with open(missing_test_path, "r") as file:
    reader = csv.reader(file)
    rows = list(reader)

header = rows[0]
data = rows[1:]

missing_test_df = spark.createDataFrame(data, schema=header)

missing_test_report = validate_schema(
    missing_test_df,
    customer_schema
)

print(missing_test_report)

In [0]:
misspelled_test_path = "/Workspace/Users/lkongara31@gmail.com/data-engineering-practice/tests/day02/customers_misspelled_column.csv"

with open(misspelled_test_path, "r") as file:
    reader = csv.reader(file)
    rows = list(reader)

header = rows[0]
data = rows[1:]

misspelled_test_df = spark.createDataFrame(data, schema=header)

misspelled_test_report = validate_schema(
    misspelled_test_df,
    customer_schema
)

print(misspelled_test_report)

In [0]:
extra_test_path = "/Workspace/Users/lkongara31@gmail.com/data-engineering-practice/tests/day02/customers_extra_column.csv"

with open(extra_test_path, "r") as file:
    reader = csv.reader(file)
    rows = list(reader)

header = rows[0]
data = rows[1:]

extra_test_df = spark.createDataFrame(data, schema=header)

extra_test_report = validate_schema(
    extra_test_df,
    customer_schema
)

print(extra_test_report)

In [0]:
invalid_ts_path = "/Workspace/Users/lkongara31@gmail.com/data-engineering-practice/tests/day02/customers_invalid_timestamp.csv"

with open(invalid_ts_path, "r") as file:
    reader = csv.reader(file)
    rows = list(reader)

header = rows[0]
data = rows[1:]

invalid_ts_df = spark.createDataFrame(data, schema=header)

invalid_ts_report = validate_schema(
    invalid_ts_df,
    customer_schema
)

print(invalid_ts_report)

In [0]:
leading_zero_path = (
    "/Workspace/Users/lkongara31@gmail.com/"
    "data-engineering-practice/tests/day02/customers_leading_zero_id.csv"
)

with open(leading_zero_path, "r") as file:
    reader = csv.reader(file)
    rows = list(reader)

header = rows[0]
data = rows[1:]

leading_zero_df = spark.createDataFrame(data, schema=header)

leading_zero_report = validate_schema(
    leading_zero_df,
    customer_schema
)

print(leading_zero_report)

display(
    leading_zero_df.select("customer_id")
)

In [0]:
duplicate_header_path = (
    "/Workspace/Users/lkongara31@gmail.com/"
    "data-engineering-practice/tests/day02/customers_duplicate_header.csv"
)

with open(duplicate_header_path, "r") as file:
    reader = csv.reader(file)
    rows = list(reader)

header = rows[0]
data = rows[1:]

duplicate_header_df = spark.createDataFrame(
    data,
    schema=header
)

duplicate_header_report = validate_schema(
    duplicate_header_df,
    customer_schema
)

print(duplicate_header_df.columns)
print(duplicate_header_report)

In [0]:
import json
from datetime import datetime, timezone

log_dir = "/Volumes/de_prac/day01/data-files/logs"
log_file = f"{log_dir}/day02_schema_validation.log"

dbutils.fs.mkdirs(log_dir)

log_entry = {
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "test_name": "missing_column_test",
    **missing_test_report
}

dbutils.fs.put(
    log_file,
    json.dumps(log_entry, indent=2),
    overwrite=True
)

print("Validation result logged to:")
print(log_file)

In [0]:
print(dbutils.fs.head(log_file))

In [0]:
def enforce_schema_gate(report):
    if not report["can_proceed_downstream"]:
        reason = "; ".join(report["critical_errors"])
        raise ValueError(
            f"SCHEMA_VALIDATION_FAILED: {reason}"
        )

    return True


try:
    enforce_schema_gate(missing_test_report)

    print("Proceeding downstream")

except ValueError as error:
    print("DOWNSTREAM BLOCKED")
    print(error)

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType
)

wrong_type_schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("customer_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("state", StringType(), True),
    StructField("updated_at", StringType(), True)
])

wrong_type_data = [
    (123, "Alice Smith", "alice@example.com", "OH", "2026-09-27T14:30:00Z")
]

wrong_type_df = spark.createDataFrame(
    wrong_type_data,
    schema=wrong_type_schema
)

datatype_test_report = validate_schema(
    wrong_type_df,
    customer_schema
)

print(datatype_test_report)

In [0]:
blank_value_data = [
    ("", "Alice Smith", "", "OH", "")
]

blank_value_df = spark.createDataFrame(
    blank_value_data,
    schema=[
        "customer_id",
        "customer_name",
        "email",
        "state",
        "updated_at"
    ]
)

blank_value_report = validate_schema(
    blank_value_df,
    customer_schema
)

print(blank_value_report)

In [0]:
print("Columns after validation:")
print(extra_test_df.columns)

display(
    extra_test_df.select(
        "customer_id",
        "phone_number"
    )
)

In [0]:
import json

report_path = (
    "/Workspace/Users/lkongara31@gmail.com/"
    "data-engineering-practice/reports/day02/schema_validation_report.json"
)

with open(report_path, "w") as file:
    json.dump(schema_report, file, indent=2)

print("Report saved to:")
print(report_path)

In [0]:
schema_report = validate_schema(
    customer_df,
    customer_schema
)

print(schema_report)

In [0]:
import json

report_path = (
    "/Workspace/Users/lkongara31@gmail.com/"
    "data-engineering-practice/reports/day02/schema_validation_report.json"
)

with open(report_path, "w") as file:
    json.dump(schema_report, file, indent=2)

print("Report saved to:")
print(report_path)

In [0]:
assert valid_test_report["validation_status"] == "PASS"

assert missing_test_report["validation_status"] == "FAIL"
assert missing_test_report["can_proceed_downstream"] is False

assert misspelled_test_report["validation_status"] == "FAIL"
assert "state" in misspelled_test_report["missing_columns"]
assert "stat" in misspelled_test_report["extra_columns"]

assert extra_test_report["validation_status"] == "PASS"
assert "phone_number" in extra_test_report["extra_columns"]

assert invalid_ts_report["validation_status"] == "FAIL"
assert invalid_ts_report["can_proceed_downstream"] is False

assert leading_zero_report["validation_status"] == "PASS"

assert duplicate_header_report["validation_status"] == "FAIL"
assert "state" in duplicate_header_report["duplicate_headers"]

assert datatype_test_report["validation_status"] == "FAIL"

assert blank_value_report["validation_status"] == "PASS"

print("ALL DAY 2 SCHEMA VALIDATION TESTS PASSED")

In [0]:
import copy

strict_schema = copy.deepcopy(customer_schema)
strict_schema["strict_mode"] = True

strict_mode_report = validate_schema(
    extra_test_df,
    strict_schema
)

print(strict_mode_report)

In [0]:
import sys

src_path = "/Workspace/Users/lkongara31@gmail.com/data-engineering-practice/src"

if src_path not in sys.path:
    sys.path.append(src_path)

from schema_validator import validate_schema

print("schema_validator imported successfully")

In [0]:
imported_validator_report = validate_schema(
    customer_df,
    customer_schema
)

print(imported_validator_report)